In [1]:
import requests
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, pipeline
from pathlib import Path
import glob
import os
from transformers import CLIPProcessor, CLIPModel

/home/equipo/Documentos/Documentos_UC/6to_ano_1er_semestre/Sistemas_Recomendadores/recomendadores/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1778955019.576193   29167 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778955019.620898   29167 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/equipo/Documentos/Documentos_UC/6to_ano_1er_semestre/S

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

dino_id = "IDEA-Research/grounding-dino-tiny"
dino_processor = AutoProcessor.from_pretrained(dino_id)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device)

In [3]:
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
# Carga de datos
# Esta celda descarga todos los datos, y los extrae

url = 'https://drive.google.com/drive/folders/1MvRYh3SudRabydH2eAAzRq2K5Os1j41-'
output_folder = 'recsys-pf'

gdown.download_folder(url, output=output_folder, quiet=False, use_cookies=False)
zip_search_pattern = os.path.join(output_folder, '*.zip')
zip_files = glob.glob(zip_search_pattern)
image_zip_path = zip_files[0]

with zipfile.ZipFile(image_zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_folder)

In [4]:
import pandas as pd
import numpy as np

interaction_df = pd.read_csv('recsys-pf/interaction.csv')
item_info_df = pd.read_csv('recsys-pf/item_info.csv')

In [ ]:
# LLM (QWEN)
print("Cargando el modelo Qwen2.5-1.5B")
generador_qwen = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    model_kwargs={"dtype": torch.bfloat16, "load_in_4bit": True},
    device_map="auto",
)

# Definimos los Prompts del Sistema que corregimos
PROMPT_SUMMARIZER = (
    "You are an expert data annotator for computer vision. Your task is to read a noisy "
    "social media post and translate it into a concise, less than 20-word declarative "
    "description of the visual scene it implies.\n"
    "Instructions:\n"
    "- Limit the description to strictly less than 20 words.\n"
    "- Concentrate on capturing visually observable attributes (subjects, setting, clothing, objects). "
    "If they are not explicit, logically infer them from the provided text, emojis, and hashtags.\n"
    "- Refrain from using engaging, subjective, or persuasive language. Use strictly declarative and objective language."
)

PROMPT_EVALUATOR = (
    "You are a summary evaluator for data annotation in computer vision. Your task is to evaluate a "
    "social media post summary according to the following criteria:\n"
    "- The summary must be strictly less than 20 words.\n"
    "- It must encapsulate the visual elements explicitly present in the original description, or "
    "correctly infer visual elements that would logically appear in an accompanying image.\n"
    "- It must use strictly declarative and objective language.\n"
    "- You must provide feedback and revision suggestions focused solely on the presence or absence of visual elements.\n"
    "Instructions:\n"
    "- If you find the summary is too long, ask for a shorter summary.\n"
    "- If you determine that no further revisions are needed and all criteria are met, end your output "
    "with <STOP> (without any extra text)."
)

PROMPT_REFINER = (
    "You are an expert data annotator for computer vision. Your task is to refine a summary based on "
    "the provided feedback. Follow all the suggestions exactly and do not provide any additional commentary. "
    "Give only one final summary as your output.\n"
    "Instructions:\n"
    "- Limit the summary to strictly fewer than 20 words.\n"
    "- Include in your description only elements which are visually observable or logically inferable from the original description.\n"
    "- Use strictly declarative and objective language."
)

def consultar_llm(prompt_sistema, prompt_usuario, max_tokens):
    mensajes = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario}
    ]
    salida = generador_qwen(
        mensajes, 
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True
    )
    return salida[0]["generated_text"][-1]["content"].strip()

Cargando el modelo Qwen2.5-1.5B


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Device set to use cuda:0


In [ ]:
rutas = glob.glob("recsys-pf/resized_images/*.jpg")
max_iteraciones = 5 # para el llm

images_features = {}
textual_features = {}
for ruta in rutas:
    item_id = os.path.basename(ruta).replace(".jpg", "")
    image = Image.open(ruta).convert("RGB")
    # Sacar tag asociada a imagen para buscarla con dion
    item_info = item_info_df[ item_info_df["item_id"] == item_id ]
    tags = item_info["tag"].iloc[0].split("·")
    text_labels = [ [ t for  t in tags] ]
    # Quizá esto no sea tan sencillo, estas categorías pueden ser más abstractas que "vestido" o "mochila". Cosas como "comedia"

    inputs = dino_processor(images=image, text=text_labels, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = dino(**inputs)

    result_boxes = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=0.4,
        text_threshold=0.3,
        target_sizes=[image.size[::-1]]
    )

    print("\nImagen: ", item_id)
    print("Categoría buscada: ", text_labels[0])
    result_box = result_boxes[0]
    """ for box, score, labels in zip(result["boxes"], result["scores"], result["labels"]):
        box = [round(x, 2) for x in box.tolist()]
        print(f"Detected {labels} with confidence {round(score.item(), 3)} at location {box}") """
    # Recortar imagen a caja encontrada si se puede
    score, box = result_box["scores"], result_box["boxes"] 
    if score.numel() != 0:
        x_min, y_min, x_max, y_max = [int(coord) for coord in box[0].cpu().tolist()]
        image = image.crop((x_min, y_min, x_max, y_max))
    else:
        print("no se encontro objeto")
    
    # obtener features de la imagen con CLIP 
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        image_features = clip.get_image_features(**inputs)
    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    images_features[item_id] = image_features

    # Creación query con LLM 
    textos = tags + [item_info["title"].iloc[0]] + [item_info["description"].iloc[0]]
    texto = " ".join( x for x in textos if isinstance(x, str) )
    prompt_usu_sum = f"Original post: {texto}"
    resumen_actual = consultar_llm(PROMPT_SUMMARIZER, prompt_usu_sum, max_tokens=25)
    prompt_usu_eval = f"Original post: {texto}\nCurrent summary: {resumen_actual}"
    for i in range(max_iteraciones):
        feedback = consultar_llm(PROMPT_EVALUATOR, prompt_usu_eval, max_tokens=50)

        if "<STOP>" in feedback:
                break

        prompt_usu_ref = (
            f"Original post: {texto}\n"
            f"Current summary: {resumen_actual}\n"
            f"Evaluator Feedback: {feedback}"
        )
        resumen_actual = consultar_llm(PROMPT_REFINER, prompt_usu_ref, max_tokens=25)

    print(texto)
    print(resumen_actual, "\n")

    # Obtener features de texto en base a características con CLIP
    inputs = clip_processor( text=resumen_actual, return_tensors="pt", padding=True, truncation=True )
    with torch.no_grad():
        text_features = clip.get_text_features(**inputs)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)    
    textual_features[item_id] = text_features




Imagen:  i148558
Categoría buscada:  ['Dogs']


/home/equipo/Documentos/Documentos_UC/6to_ano_1er_semestre/Sistemas_Recomendadores/recomendadores/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Dogs The Legend of the Husky
Dogs Legends of the Husky 


Imagen:  i397389
Categoría buscada:  ['Celebrities Mix']
Celebrities Mix [Kim Do Young/Kong Myung] The NCT member who is closest to Brother Kong Myung is Who is the closest member of NCT besides his real brother Do Young? Let's see how brother Kong Myung answered! Cover: - Kasha SuMi Translated by daisychim Original Video: video link: https://youtu.be/cExWuKCWMGk
The pair Kim Do Young and Kong Myung are compared with respect to their relationship dynamics and comparison among NCT members, with 


Imagen:  i63014
Categoría buscada:  ['Animals Mix']
Animals Mix Picked up a super sticky cute little puppy, lively not barking, too cute Picked up a super sticky cute little puppy, lively not barking, is too cute!
Animals mixed with a super sticky, cute little puppy, lively but not barking, very cute. 


Imagen:  i30828
Categoría buscada:  ['Comedy']
no se encontro objeto
Comedy The first thing you need to do is to get your hands on som

KeyboardInterrupt: 

In [ ]:
# Faltaría el codigo que hace la recomendación
interaction_df

,item_id,user_id,timestamp
0,i72138,u209296,1605059546
1,i15530,u2444520,1628914341
2,i95199,u1866870,1601008921
3,i3413,u2498546,1505731122
4,i224963,u3676118,1643894394
...,...,...,...
989489,i53095,u5426144,1648142361
989490,i219825,u2841982,1605073212
989491,i42184,u2257424,1644932293
989492,i84962,u1879882,1639377159
